In [1]:
from youtube_transcript_api import YouTubeTranscriptApi
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from dotenv import load_dotenv
import random
import os
import json

load_dotenv(override=True)
groq_api_key = os.getenv("GROQ_API_KEY")

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.7,
    api_key=groq_api_key
)

In [2]:
def get_video_id(url):
    if "v=" in url:
        video_id = url.split("v=")[1]
        video_id = video_id.split("&")[0]
        return video_id
    elif "youtu.be/" in url:
        video_id = url.split("youtu.be/")[1]
        video_id = video_id.split("?")[0]
        return video_id
    else:
        return None

def get_transcript(url):
    video_id = get_video_id(url)
    if not video_id:
        print("Invalid Youtube URL")
        return None
    try:
        ytt_api = YouTubeTranscriptApi()
        transcript = ytt_api.fetch(video_id)
        return transcript
    except Exception as e:
        print("Error: Could not fetch transcript. The video may not have captions or may not exist.")
        return None

def chunk_transcript(transcript, chunk_duration=120):
    chunks = []
    current_text = ""
    chunk_start = transcript[0].start

    for segment in transcript:
        if segment.start - chunk_start >= chunk_duration:
            chunks.append({
                "text": current_text.strip(),
                "start": chunk_start,
                "end": segment.start
            })
            current_text = segment.text + " "
            chunk_start = segment.start
        else:
            current_text += segment.text + " "

    # Don't forget the last chunk that's still being built
    if current_text:
        chunks.append({
            "text": current_text.strip(),
            "start": chunk_start,
            "end": transcript[-1].start
        })

    return chunks


class TranscriptionAgent:
    def __init__(self):
        self.name = "Transcription Agent"
    
    def run(self, url):
        transcript = get_transcript(url)
        if not transcript:
            return None
        chunks = chunk_transcript(transcript)
        return chunks

In [3]:
class FilterAgent:
    def __init__(self, llm):
        self.name = "Filter Agent"
        self.llm = llm
    
    def run(self, chunks):
        educational_chunks = []
        for chunk in chunks:
            try:
                prompt = f"""Look at this transcript section and decide if it contains actual educational content or if it is filler content.

Filler content includes: introductions, outros, subscribe reminders, agenda overview, quiz segments, promotions, greetings, channel promotions, "like and share" requests, or any non-teaching content.

IMPORTANT: If the section contains ANY educational explanation, concepts, or teaching, even if mixed with casual talk, mark it as "educational".
Only mark as "skip" if the ENTIRE section is filler with NO educational value.

Transcript:
{chunk['text']}

Reply with ONLY one word: "educational" or "skip"
"""
                response = self.llm.invoke(prompt)
                content = response.content.strip().lower()
                if content == "educational":
                    educational_chunks.append(chunk)
                else:
                    print(f"Skipping filler chunk at {chunk['start']:.0f}s")
            except Exception as e:
                print(f"Error filtering chunk: {e}")
                educational_chunks.append(chunk)
        
        print(f"Kept {len(educational_chunks)} out of {len(chunks)} chunks")
        return educational_chunks
        # For each chunk:
        #   Ask LLM if it's educational or filler
        #   If educational, keep it
        #   If filler, skip it
        # Return only educational chunks

In [4]:
class ConceptExtractionAgent:
    def __init__(self, llm):
        self.name = "Concept Extraction Agent"
        self.llm = llm
    
    def run(self, chunks):
        all_concepts = []
        
        for chunk in chunks:
            try:
                start_min = int(chunk["start"] // 60)
                start_sec = int(chunk["start"] % 60)
                end_min = int(chunk["end"] // 60)
                end_sec = int(chunk["end"] % 60)
                timestamp = f"{start_min}:{start_sec:02d} - {end_min}:{end_sec:02d}"
                
                prompt = f"""Extract the 2-3 most important concepts or topics from this transcript section.

Transcript:
{chunk['text']}

Return ONLY JSON in this format:
{{"concepts": ["concept 1", "concept 2", "concept 3"]}}
"""
                response = self.llm.invoke(prompt)
                content = response.content.strip()
                if content.startswith("```json"):
                    content = content[7:]
                if content.startswith("```"):
                    content = content[3:]
                if content.endswith("```"):
                    content = content[:-3]
                content = content.strip()
                
                concepts = json.loads(content)
                concepts["timestamp"] = timestamp
                all_concepts.append(concepts)
            except Exception as e:
                print(f"Skipping chunk: {e}")
                continue
        
        return all_concepts

In [5]:
class QuestionGenerationAgent:
    def __init__(self, llm):
        self.name = "Question Generation Agent"
        self.llm = llm

    def shuffle_options(self, question):
        options = list(question["options"].items())
        random.shuffle(options)
    
        old_correct = question["correct_answer"]
        correct_text = question["options"][old_correct]
    
        new_options = {}
        new_correct = None
        for i, (_, text) in enumerate(options):
            letter = ["A", "B", "C", "D"][i]
            new_options[letter] = text
            if text == correct_text:
                new_correct = letter
    
        question["options"] = new_options
        question["correct_answer"] = new_correct
        return question
    
    def run(self, concepts_list):
        all_questions = []
        
        for item in concepts_list:
            try:
                concepts = ", ".join(item["concepts"])
                timestamp = item["timestamp"]
                
                prompt = f"""Generate 1 multiple choice question that tests understanding of these concepts: {concepts}

The question should have:
- 4 options (A, B, C, D)
- The correct answer
- An explanation of why that answer is correct

Return ONLY JSON in this format:
{{
    "question": "the question text",
    "options": {{"A": "...", "B": "...", "C": "...", "D": "..."}},
    "correct_answer": "B",
    "explanation": "why this is correct"
}}
"""
                response = self.llm.invoke(prompt)
                content = response.content.strip()
                if content.startswith("```json"):
                    content = content[7:]
                if content.startswith("```"):
                    content = content[3:]
                if content.endswith("```"):
                    content = content[:-3]
                content = content.strip()
                
                question = json.loads(content)
                question = self.shuffle_options(question)
                question["timestamp"] = timestamp
                question["concepts"] = item["concepts"]
                all_questions.append(question)
            except Exception as e:
                print(f"Skipping question: {e}")
                continue
        
        return {"questions": all_questions}

In [6]:
class EvaluationAgent:
    def __init__(self, llm):
        self.name = "Evaluation Agent"
        self.llm = llm

    def deduplicate_topics(self, weak_topics):
        prompt = f"""Here is a list of topics:
{weak_topics}

Group similar or related topics together and give me a clean, 
deduplicated list. Combine topics that mean the same thing into 
one clear topic name.

Return ONLY JSON in this format:
{{"topics": ["topic 1", "topic 2", "topic 3"]}}
"""
        try:
            response = self.llm.invoke(prompt)
            content = response.content.strip()
            if content.startswith("```json"):
                content = content[7:]
            if content.startswith("```"):
                content = content[3:]
            if content.endswith("```"):
                content = content[:-3]
            content = content.strip()
        
            result = json.loads(content)
            return result["topics"]
        except Exception as e:
            print(f"Deduplication failed: {e}")
            return weak_topics
    
    def run(self, quiz_data):
        score = 0
        weak_topics = []
        
        for i, q in enumerate(quiz_data["questions"]):
            print(f"\nQuestion {i+1}: {q['question']}")
            for option, text in q["options"].items():
                print(f"  {option}) {text}")
            user_answer = input("\nYour answer (A/B/C/D): ").upper()
            if user_answer == q['correct_answer']:
                score = score + 1
                print("Correct!")
            else:
                print("Incorrect!")
                weak_topics.extend(q['concepts'])
                weak_topics = list(set(weak_topics))
            print(f"Correct Answer: {q['correct_answer']}")
            print(f"Explanation: {q['explanation']}")
            print(f"Review in video at: {q['timestamp']}")
            
        print(f"\nFinal Score: {score}/{len(quiz_data['questions'])}")
        if weak_topics:
            weak_topics = self.deduplicate_topics(weak_topics)
        print(f"\nWeak Topics: {weak_topics}")

        return {
            "score": score,
            "total": len(quiz_data["questions"]),
            "weak_topics": weak_topics
        }

In [ ]:
# Agent 1: Get transcript
agent1 = TranscriptionAgent()
chunks = agent1.run("https://www.youtube.com/watch?v=aircAruvnKk")
print(f"Agent 1 done: {len(chunks)} chunks")

# Agent 2: Extract concepts
agent2 = ConceptExtractionAgent(llm)
concepts = agent2.run(chunks)
print(f"Agent 2 done: {len(concepts)} concept sets")

# Agent 3: Generate questions
agent3 = QuestionGenerationAgent(llm)
quiz = agent3.run(concepts)
print(f"Agent 3 done: {len(quiz['questions'])} questions")

# Agent 4: Evaluate
agent4 = EvaluationAgent(llm)
results = agent4.run(quiz)

In [7]:
def run_pipeline(url, num_questions=10):
    # Agent 1
    agent1 = TranscriptionAgent()
    chunks = agent1.run(url)
    if not chunks:
        return None
    print(f"Agent 1 done: {len(chunks)} chunks")

    # Agent 1.5: Filter
    agent_filter = FilterAgent(llm)
    filtered_chunks = agent_filter.run(chunks)
    if not filtered_chunks:
        return None
    
    # Select chunks from filtered ones
    total_chunks = len(filtered_chunks)
    if total_chunks <= num_questions:
        selected_chunks = filtered_chunks
    else:
        step = total_chunks // num_questions
        selected_chunks = [filtered_chunks[i * step] for i in range(num_questions)]
    print(f"Selected {len(selected_chunks)} chunks for quiz")
    
    # Agent 2: Extract concepts
    agent2 = ConceptExtractionAgent(llm)
    concepts = agent2.run(selected_chunks)
    if not concepts:
        return None
    else:
        print(f"Agent 2 done: {len(concepts)} concept sets")
     
    # Agent 3
    agent3 = QuestionGenerationAgent(llm)
    quiz = agent3.run(concepts)
    if not quiz:
        return None
    else:
        print(f"Agent 3 done: {len(quiz['questions'])} questions")
    
    # Agent 4
    agent4 = EvaluationAgent(llm)
    results = agent4.run(quiz)

    # Ask if user wants adaptive quiz
    if results["weak_topics"]:
        take_adaptive = input("\nYou have weak topics. Want to take an adaptive quiz? (yes/no): ").lower()
        if take_adaptive == "yes":
            adaptive_results = run_adaptive_quiz(results["weak_topics"], filtered_chunks)
            results["adaptive_results"] = adaptive_results
    
    return results
    # Return results
    results["filtered_chunks"] = filtered_chunks
    return results

In [8]:
def run_adaptive_quiz(weak_topics, filtered_chunks):
    matching_chunks = []
    
    for chunk in filtered_chunks:
        prompt = f"""Here are the topics the student is weak at:
{weak_topics}
Does this transcript section cover any of these topics?
Transcript:
{chunk['text']}
Reply with ONLY one word: "yes" or "no"
"""
        try:
            response = llm.invoke(prompt)
            content = response.content.strip().lower()
            if content == "yes":
                matching_chunks.append(chunk)
        except Exception as e:
            print(f"Error matching chunk: {e}")
            continue
    
    print(f"Found {len(matching_chunks)} chunks matching weak topics")

    # Limit to 5 chunks to save API calls
    if len(matching_chunks) > 5:
        step = len(matching_chunks) // 5
        matching_chunks = [matching_chunks[i * step] for i in range(5)]
        print(f"Selected 5 chunks for adaptive quiz")
    
    if not matching_chunks:
        print("No matching chunks found")
        return None
    
    # Agent 2
    agent2 = ConceptExtractionAgent(llm)
    concepts = agent2.run(matching_chunks)
    if not concepts:
        return None
    print(f"Agent 2 done: {len(concepts)} concept sets")
    
    # Agent 3
    agent3 = QuestionGenerationAgent(llm)
    quiz = agent3.run(concepts)
    if not quiz:
        return None
    print(f"Agent 3 done: {len(quiz['questions'])} questions")
    
    # Agent 4
    agent4 = EvaluationAgent(llm)
    results = agent4.run(quiz)
    
    return results

In [9]:
#results = run_pipeline("https://www.youtube.com/watch?v=_gGRBWaRpAQ&list=PLZoTAELRMXVNNrHSKv36Lr3_156yCo6Nn&index=5")

# Round 1
results = run_pipeline("https://www.youtube.com/watch?v=_gGRBWaRpAQ&list=PLZoTAELRMXVNNrHSKv36Lr3_156yCo6Nn&index=5")

# Round 2
adaptive_results = run_adaptive_quiz(results["weak_topics"], results["filtered_chunks"])

Agent 1 done: 37 chunks
Skipping filler chunk at 14s
Skipping filler chunk at 134s
Skipping filler chunk at 254s
Skipping filler chunk at 3295s
Skipping filler chunk at 3415s
Skipping filler chunk at 3537s
Skipping filler chunk at 3780s
Skipping filler chunk at 4146s
Skipping filler chunk at 4267s
Skipping filler chunk at 4391s
Kept 27 out of 37 chunks
Selected 10 chunks for quiz
Agent 2 done: 10 concept sets
Agent 3 done: 10 questions

Question 1: What is the primary purpose of removing stop words during text preprocessing in a Bag of Words model?
  A) To convert all words to their base form
  B) To eliminate common words like 'the' and 'and' that do not add much value to the meaning of the text
  C) To improve the spelling of words in the text
  D) To reduce the dimensionality of the feature space by removing rare words



Your answer (A/B/C/D):  b


Correct!
Correct Answer: B
Explanation: Stop words are common words like 'the', 'and', 'a', etc. that do not carry much meaning in a sentence. Removing them helps in reducing noise and improving the model's performance by focusing on more important words. This is a key step in text preprocessing for Bag of Words models.
Review in video at: 6:21 - 8:23

Question 2: When dealing with out-of-vocabulary words in a natural language processing model that uses vector representation, which of the following is a common approach to handle them without significantly increasing the dimensionality of the sparse matrix used to store word embeddings?
  A) Use a special token to represent all out-of-vocabulary words
  B) Train a new model for each out-of-vocabulary word
  C) Remove out-of-vocabulary words from the dataset
  D) Increase the dimensionality of the sparse matrix to accommodate all possible words



Your answer (A/B/C/D):  c


Incorrect!
Correct Answer: A
Explanation: This is correct because using a special token to represent all out-of-vocabulary words allows the model to handle unseen words without increasing the dimensionality of the sparse matrix. This approach is efficient and effective, as it groups all unknown words together and assigns them a single vector representation, thereby avoiding the need to add new dimensions to the matrix for each new word.
Review in video at: 10:24 - 12:26

Question 3: Which of the following techniques is most suitable for comparing the semantic meaning of two documents based on their content, taking into account the importance of each word in the entire corpus?
  A) Inverse Document Frequency only
  B) Term Frequency only
  C) Combining TF-IDF with Cosine Similarity
  D) Cosine Similarity alone


...

Incorrect!
Correct Answer: C
Explanation: Stop word removal is a common step in text preprocessing for a Bag of Words model. It involves removing frequently occurring words like 'the', 'and', etc. that do not add much value to the meaning of the text. This helps reduce the dimensionality of the feature space and improves the model's performance. Tokenization, stemming, and lemmatization are also important steps in text preprocessing, but they serve different purposes.
Review in video at: 6:21 - 8:23

Question 2: What is the primary goal of text preprocessing in the context of creating a Bag of Words model, which relies heavily on word frequency to represent text data?
  A) To increase the frequency of rare words in the text data
  B) To tokenize the text data into sentences instead of individual words
  C) To reduce the dimensionality of the text data by removing infrequent words
  D) To normalize the text data by converting all words to lowercase and removing punctuation and stop word


Your answer (A/B/C/D):  m


Incorrect!
Correct Answer: D
Explanation: The primary goal of text preprocessing in the context of creating a Bag of Words model is to normalize the text data. This involves converting all words to lowercase to ensure the model treats the same words with different cases as a single word, removing punctuation to prevent it from being treated as part of a word, and removing stop words (common words like 'the', 'and', etc.) that do not add much value to the meaning of the text. This normalization helps in creating a more accurate representation of word frequencies, which is crucial for the Bag of Words model.
Review in video at: 8:23 - 10:24

Question 3: What does cosine similarity measure between two vector representations of words in a high-dimensional space?
  A) The dot product of the vectors without considering their magnitudes
  B) The magnitude of the difference between the vectors
  C) The cosine of the angle between the two vectors, indicating their semantic similarity
  D) The E


Your answer (A/B/C/D):  m


Incorrect!
Correct Answer: C
Explanation: Cosine similarity is a measure used to calculate the similarity between two vectors by finding the cosine of the angle between them. This is particularly useful in natural language processing for comparing the semantic meaning of words represented as vectors in a high-dimensional space. It effectively captures how similar the directions of the vectors are, which corresponds to the similarity in meaning between the words they represent, regardless of their magnitude.
Review in video at: 12:26 - 14:26

Final Score: 0/3
Deduplication failed: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kpy97fwrfcm8crm26tdman33` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99971, Requested 131. Please try again in 1m28.128s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

Weak